In [2]:
pip install pandas numpy scikit-learn xgboost catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    roc_auc_score, f1_score, recall_score, precision_score, fbeta_score
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ====================================================
# 1. LOAD & INITIAL CLEANING
# ====================================================
print("📂 Loading data and performing initial cleanup...")
df = pd.read_csv("hello.csv")
df = df.loc[:, ~df.columns.duplicated()].copy()

df["requires_road_closure"] = (
    df["requires_road_closure"]
    .astype(str)
    .str.lower()
    .map({"true": 1, "false": 0, "yes": 1, "no": 0})
)
df = df.dropna(subset=["requires_road_closure", "start_datetime"]).reset_index(drop=True)
df["requires_road_closure"] = df["requires_road_closure"].astype(int)

df["start_datetime"] = pd.to_datetime(df["start_datetime"], errors="coerce")
df = df.sort_values("start_datetime").reset_index(drop=True)

# ====================================================
# 2. ANTI-LEAKAGE RESET
# ====================================================
leak_cols = ["end_datetime", "resolved_at_latitude", "resolved_at_longitude", "status", "duration_minutes"]
df = df.drop(columns=[c for c in leak_cols if c in df.columns], errors="ignore")

# ====================================================
# 3. PRODUCTION FEATURE ENGINEERING & HOTSPOT DISTANCE
# ====================================================
df["hour"] = df["start_datetime"].dt.hour
df["day_of_week"] = df["start_datetime"].dt.dayofweek
df["month"] = df["start_datetime"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)

df["has_emergency_keywords"] = df["description"].fillna("").astype(str).str.lower().apply(
    lambda x: int(any(k in x for k in ["accident", "fire", "blocked", "fallen", "injury"]))
)

vehicle_risk_map = {"motor_cycle": 1, "car": 2, "taxi": 2, "bus": 4, "truck": 5, "heavy_vehicle": 6}
df["vehicle_risk"] = df["veh_type"].astype(str).str.lower().map(vehicle_risk_map).fillna(3)
df["urgency_score"] = df["priority"].fillna("Low").astype(str).map({"High": 3, "Medium": 2, "Low": 1}).fillna(1) * df["vehicle_risk"]

# Spatial Anchor Clustering
coords = df[["latitude", "longitude"]].fillna(0)
kmeans = KMeans(n_clusters=25, random_state=42, n_init=10)
df["geo_cluster"] = kmeans.fit_predict(coords)

# UPGRADE B: Vectorized Distance to Cluster Hotspot Center (No heavy row-by-row applies)
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=["center_lat", "center_lon"])
df = df.merge(cluster_centers, left_on="geo_cluster", right_index=True, how="left")
df["dist_to_hotspot"] = np.sqrt((df["latitude"] - df["center_lat"])**2 + (df["longitude"] - df["center_lon"])**2)
df = df.drop(columns=["center_lat", "center_lon"])

# ====================================================
# 4. CHRONOLOGICAL SPLIT
# ====================================================
split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# UPGRADE C: Calculate Class Weights strictly from historical training distribution
num_neg = (train_df["requires_road_closure"] == 0).sum()
num_pos = (train_df["requires_road_closure"] == 1).sum()
calculated_ratio = num_neg / max(1, num_pos)

# ====================================================
# 5. LEAKAGE-FREE TARGET ENCODING
# ====================================================
def kfold_target_encoding(train, test, col, target, n_splits=5, alpha=12):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    global_mean = train[target].mean()
    train_encoded = pd.Series(index=train.index, dtype=float)

    for tr_idx, val_idx in skf.split(train, train[target]):
        tr = train.iloc[tr_idx]
        stats = tr.groupby(col)[target].agg(['count', 'mean'])
        smooth = (stats['count'] * stats['mean'] + alpha * global_mean) / (stats['count'] + alpha)
        train_encoded.iloc[val_idx] = train.iloc[val_idx][col].map(smooth).fillna(global_mean)

    stats_full = train.groupby(col)[target].agg(['count', 'mean'])
    smooth_full = (stats_full['count'] * stats_full['mean'] + alpha * global_mean) / (stats_full['count'] + alpha)
    test_encoded = test[col].map(smooth_full).fillna(global_mean)
    return train_encoded, test_encoded

risk_cols = ["event_cause", "corridor", "junction", "police_station"]
for col in risk_cols:
    train_df[col+"_risk"], test_df[col+"_risk"] = kfold_target_encoding(train_df, test_df, col, "requires_road_closure")

train_df["congestion_potential"] = train_df["vehicle_risk"] * train_df["corridor_risk"]
test_df["congestion_potential"] = test_df["vehicle_risk"] * test_df["corridor_risk"]

# ====================================================
# 6. CATEGORICAL LABEL ENCODING
# ====================================================
cat_cols = ["event_type", "event_cause", "veh_type", "corridor", "priority", "police_station", "zone", "junction"]
cat_cols = [
    "event_type",
    "event_cause",
    "veh_type",
    "corridor",
    "priority",
    "police_station",
    "zone",
    "junction"
]

encoders = {}

for col in cat_cols:

    train_df[col] = train_df[col].fillna("Unknown").astype(str)
    test_df[col] = test_df[col].fillna("Unknown").astype(str)

    encoder = LabelEncoder()
    encoder.fit(pd.concat([train_df[col], test_df[col]]))

    train_df[col] = encoder.transform(train_df[col])
    test_df[col] = encoder.transform(test_df[col])

    encoders[col] = encoder

import joblib
joblib.dump(encoders, "encoders1.pkl")

# ====================================================
# 7. FINAL VALIDATED FEATURE POOL
# ====================================================
features = [
    "event_type", "event_cause", "veh_type", "corridor", "priority", 
    "police_station", "zone", "junction", "latitude", "longitude", 
    "hour", "day_of_week", "month", "is_weekend", "hour_sin", "hour_cos",
    "vehicle_risk", "event_cause_risk", "corridor_risk", "junction_risk", 
    "police_station_risk", "congestion_potential",
    "has_emergency_keywords", "urgency_score", "geo_cluster", "dist_to_hotspot"
]

X_train = train_df[features]
X_test = test_df[features]
y_train = train_df["requires_road_closure"].astype(int).values
y_test = test_df["requires_road_closure"].astype(int).values

# ====================================================
# 8. OUT-OF-FOLD BLENDER ENGINE WITH INSIDE-FOLD WEIGHTING
# ====================================================
print("⏳ Generating Unbiased Calibrated OOF Predictions...")
xgb_oof = np.zeros(len(X_train))
cat_oof = np.zeros(len(X_train))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cat_features_idx = [features.index(col) for col in cat_cols if col in features]

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
    
    # Calculate native fold ratio dynamically to prevent fold-level leakage
    fold_ratio = (y_tr == 0).sum() / max(1, (y_tr == 1).sum())
    
    xgb_fold = XGBClassifier(
        n_estimators=250, max_depth=4, learning_rate=0.02, 
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=fold_ratio,
        random_state=42, eval_metric="logloss"
    )
    cat_fold = CatBoostClassifier(
        iterations=300, depth=5, learning_rate=0.03, 
        auto_class_weights='Balanced', cat_features=cat_features_idx, random_seed=42, verbose=0
    )
    
    xgb_fold.fit(X_tr, y_tr)
    cat_fold.fit(X_tr, y_tr)
    
    xgb_oof[val_idx] = xgb_fold.predict_proba(X_va)[:, 1]
    cat_oof[val_idx] = cat_fold.predict_proba(X_va)[:, 1]

print("🚀 Fitting final base estimators on full historical set...")
xgb_base = XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=calculated_ratio, random_state=42, eval_metric="logloss")
cat_base = CatBoostClassifier(iterations=300, depth=5, learning_rate=0.03, auto_class_weights='Balanced', cat_features=cat_features_idx, random_seed=42, verbose=0)

xgb_base.fit(X_train, y_train)
cat_base.fit(X_train, y_train)

stack_train = np.column_stack([xgb_oof, cat_oof])
stack_test = np.column_stack([xgb_base.predict_proba(X_test)[:, 1], cat_base.predict_proba(X_test)[:, 1]])

# UPGRADE A: Probability Calibration Layer on Stacking Level
print("🎯 Calibrating Meta-Classifier probabilities via Isotonic Regression...")
base_meta = LogisticRegression(random_state=42)
meta_model = CalibratedClassifierCV(estimator=base_meta, method="isotonic", cv=3)
meta_model.fit(stack_train, y_train)
prob = meta_model.predict_proba(stack_test)[:, 1]

# ====================================================
# 9. OPTIMIZATION & PERFORMANCE REPORT
# ====================================================
best_threshold = 0.5
best_score = 0
for t in np.arange(0.1, 0.9, 0.01):
    current_preds = (prob >= t).astype(int)
    score = fbeta_score(y_test, current_preds, beta=2, zero_division=0)
    if score > best_score:
        best_score = score
        best_threshold = t

pred = (prob >= best_threshold).astype(int)

print("\n==============================================")
print(f"🛡️ PRODUCTION-GRADE CALIBRATED STACKED ENGINE (Threshold: {round(best_threshold, 2)})")
print("==============================================")
print("Accuracy  :", round(accuracy_score(y_test, pred), 4))
print("Precision :", round(precision_score(y_test, pred), 4))
print("Recall    :", round(recall_score(y_test, pred), 4))
print("F1-Score  :", round(f1_score(y_test, pred), 4))
print("ROC AUC   :", round(roc_auc_score(y_test, prob), 4))

📂 Loading data and performing initial cleanup...
⏳ Generating Unbiased Calibrated OOF Predictions...
🚀 Fitting final base estimators on full historical set...
🎯 Calibrating Meta-Classifier probabilities via Isotonic Regression...

🛡️ PRODUCTION-GRADE CALIBRATED STACKED ENGINE (Threshold: 0.14)
Accuracy  : 0.8183
Precision : 0.3571
Recall    : 0.7143
F1-Score  : 0.4762
ROC AUC   : 0.8383


In [21]:
print(encoders["event_cause"].classes_[:20])

['Debris' 'Fog / Low Visibility' 'accident' 'congestion' 'construction'
 'debris' 'others' 'pot_holes' 'procession' 'protest' 'public_event'
 'road_conditions' 'test_demo' 'tree_fall' 'vehicle_breakdown'
 'vip_movement' 'water_logging']


In [23]:
import joblib

joblib.dump(features, "features.pkl")

['features.pkl']

In [4]:
import joblib

joblib.dump(xgb_base, "xgb_model.pkl")
joblib.dump(cat_base, "cat_model.pkl")
joblib.dump(meta_model, "meta_model.pkl")
joblib.dump(kmeans, "kmeans.pkl")
joblib.dump(best_threshold, "best_threshold.pkl")

print("Models Saved Successfully")

Models Saved Successfully


In [39]:
risk_lookup = {}

for col in [
    "event_cause",
    "corridor",
    "junction",
    "police_station"
]:
    risk_lookup[col] = (
        train_df.groupby(col)["requires_road_closure"]
        .mean()
        .to_dict()
    )

import joblib
joblib.dump(risk_lookup, "risk_lookup_new.pkl")

['risk_lookup_new.pkl']